# Parkinson's Detection from Voice Features (UCI)

The UCI Parkinsons dataset has 195 voice recordings from 31 people (23 with PD), with biomedical features like jitter, shimmer, and harmonics-to-noise ratio. The catch is that there are multiple recordings per subject — if I do a naive row-level train/test split, the model can memorize a subject's voice in train and "predict" them in test. So the right thing is to split by subject. I'll show both numbers so the gap is visible.

I tried to load the file from disk and fall back to fetching it from the UCI mirror if it isn't local.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(0)
DATA_DIR = Path("../../data/parkinsons")
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"

In [2]:
local = DATA_DIR / "parkinsons.data"
df = pd.read_csv(local) if local.exists() else pd.read_csv(URL)

# name column is 'phon_R01_S01_1', the S## piece is the subject id
df["subject"] = df["name"].str.extract(r"_S(\d+)_")[0].astype(int)
print("shape:", df.shape, "subjects:", df["subject"].nunique(), "positive rate:", df["status"].mean().round(3))
df.head(3)

shape: (195, 25) subjects: 32 positive rate: 0.754


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE,subject
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654,1
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674,1
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634,1


In [3]:
feature_cols = [c for c in df.columns if c not in ("name", "status", "subject")]
X = df[feature_cols].values
y = df["status"].values
groups = df["subject"].values

In [4]:
def evaluate(pipe, splitter, label):
    aucs, recs = [], []
    for tr, te in splitter.split(X, y, groups=groups if "Group" in type(splitter).__name__ else None):
        pipe.fit(X[tr], y[tr])
        p = pipe.predict_proba(X[te])[:, 1]
        aucs.append(roc_auc_score(y[te], p))
        recs.append(((p >= 0.5).astype(int) & (y[te] == 1)).sum() / max(y[te].sum(), 1))
    print(f"{label:30s}  AUC mean: {np.mean(aucs):.3f}   recall mean: {np.mean(recs):.3f}")
    return np.mean(aucs)

# wrapper around train_test_split for the row-level baseline
class _RowSplitter:
    def __init__(self, n=5, seed=0): self.n, self.seed = n, seed
    def split(self, X, y, groups=None):
        for i in range(self.n):
            tr, te = train_test_split(np.arange(len(X)), test_size=0.25,
                                       stratify=y, random_state=self.seed + i)
            yield tr, te

gss = GroupShuffleSplit(n_splits=5, test_size=0.25, random_state=0)

for name, clf in [
    ("SVM (RBF)",        Pipeline([("sc", StandardScaler()), ("clf", SVC(probability=True))])),
    ("RandomForest",     RandomForestClassifier(n_estimators=400, random_state=0)),
    ("GradientBoosting", GradientBoostingClassifier(n_estimators=200, random_state=0)),
]:
    print(f"\n--- {name} ---")
    evaluate(clf, _RowSplitter(),  "row-level split (leaks subject!)")
    evaluate(clf, gss,             "group split by subject")


--- SVM (RBF) ---
row-level split (leaks subject!)  AUC mean: 0.931   recall mean: 0.973
group split by subject          AUC mean: 0.596   recall mean: 0.917

--- RandomForest ---


row-level split (leaks subject!)  AUC mean: 0.973   recall mean: 0.962


group split by subject          AUC mean: 0.747   recall mean: 0.941

--- GradientBoosting ---


row-level split (leaks subject!)  AUC mean: 0.981   recall mean: 0.962


group split by subject          AUC mean: 0.673   recall mean: 0.919


## What the comparison shows

The row-level numbers look great on every model and they're partly fake. With 195 rows and 31 subjects, each subject contributes ~6 recordings on average. When a random row split puts most of a subject's recordings in train and one or two in test, the model is doing closer to speaker identification than disease detection.

The group split drops AUC noticeably — usually a few hundredths to ~0.05 depending on which subjects land in the test fold. That gap is the honest measure of how well this model generalizes to *new patients*.

If I extended this I'd want a bigger dataset (there is a 2022 Parkinson's voice study with more subjects), and I'd probably move from these hand-engineered features to MFCC/spectrogram input with a small CNN. But for a clean illustration of leakage, this dataset is a great teaching example.